# phast — Quick Start (Colab)

<a href="https://colab.research.google.com/github/CEMS-Lab/PhAST/blob/main/notebooks/quickstart_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

`phast` is a matrix-free, auto-differentiable phase-field fracture solver written entirely in PyTorch. This notebook walks through the building blocks of the solver — sparse-direct linear solve with autograd, a small linear-elastic plate, the Hulbert–Chung generalized-α dynamic integrator, and the mixed-precision CG iterative solver — so you can try the engine end-to-end in your browser.

Runtime: CPU is fine for everything below. GPU optional.

## 1. Install

We `pip install` the package and also `git clone` the repo so the example scripts are available on disk.

In [ ]:
!pip install -q git+https://github.com/CEMS-Lab/PhAST.git
!git clone -q https://github.com/CEMS-Lab/PhAST.git /content/phast_repo || true
import sys
sys.path.insert(0, '/content/phast_repo')

import torch
print(f"Python {sys.version.split()[0]}, torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

## 2. Sparse-solve walkthrough

`SparseSolveAutograd` wraps SciPy SuperLU as a `torch.autograd.Function`. The forward solves $K x = b$, the backward reuses the LU factor to solve $K^\top \lambda = \partial L/\partial x$ and returns the implicit-function gradient `grad_K_values[k] = -lambda[i_k] * x[j_k]`, `grad_b = lambda`.

Inputs are `(K_indices: (2,nnz) int64 COO, K_values: (nnz,) float64, b: (n,) float64, n: int)`.

In [ ]:
import torch
from sparse_solve import SparseSolveAutograd

torch.manual_seed(0)
n = 5
# Build a small SPD tridiagonal K = tridiag(-1, 2, -1)
rows = torch.cat([torch.arange(n), torch.arange(n-1), torch.arange(1, n)])
cols = torch.cat([torch.arange(n), torch.arange(1, n), torch.arange(n-1)])
vals = torch.cat([2.0*torch.ones(n, dtype=torch.float64),
                  -torch.ones(n-1, dtype=torch.float64),
                  -torch.ones(n-1, dtype=torch.float64)])
vals.requires_grad_(True)
indices = torch.stack([rows, cols])
b = torch.ones(n, dtype=torch.float64, requires_grad=True)

x = SparseSolveAutograd.apply(indices, vals, b, n)
print('x =', x.detach().numpy())

# Autograd: d(sum x)/d b should equal lambda from K^T lambda = 1
loss = x.sum()
loss.backward()
print('grad_b =', b.grad.numpy())
print('grad_K_values (first 5) =', vals.grad[:5].numpy())

## 3. Linear elastic plate

A clamped–tip-loaded CST cantilever, assembled in pure torch, solved through `SparseSolveAutograd`, with `dE` recovered through the solve. Mirrors `examples/solid_mechanics/linear_plate/run.py`.

In [ ]:
import torch
from sparse_solve import SparseSolveAutograd
torch.set_default_dtype(torch.float64)

def build_mesh(nx, ny, L, H):
    xs = torch.linspace(0., L, nx+1); ys = torch.linspace(0., H, ny+1)
    X, Y = torch.meshgrid(xs, ys, indexing='ij')
    coords = torch.stack([X.reshape(-1), Y.reshape(-1)], dim=1)
    nid = lambda i, j: i*(ny+1)+j
    elems = []
    for i in range(nx):
        for j in range(ny):
            n00, n10, n11, n01 = nid(i,j), nid(i+1,j), nid(i+1,j+1), nid(i,j+1)
            elems += [[n00,n10,n11], [n00,n11,n01]]
    return coords, torch.tensor(elems, dtype=torch.long)

def D_plane_strain(E, nu):
    c = E / ((1+nu)*(1-2*nu))
    D = torch.zeros(3,3, dtype=E.dtype if E.ndim else torch.float64)
    D[0,0]=c*(1-nu); D[1,1]=c*(1-nu); D[0,1]=c*nu; D[1,0]=c*nu; D[2,2]=c*(0.5-nu)
    return D

def cst_BA(ce):
    (x1,y1),(x2,y2),(x3,y3) = ce[0], ce[1], ce[2]
    A2 = (x2-x1)*(y3-y1) - (x3-x1)*(y2-y1); A = 0.5*A2
    b = torch.stack([y2-y3, y3-y1, y1-y2]) / A2
    c = torch.stack([x3-x2, x1-x3, x2-x1]) / A2
    B = torch.zeros(3,6, dtype=ce.dtype)
    for k in range(3):
        B[0,2*k]=b[k]; B[1,2*k+1]=c[k]; B[2,2*k]=c[k]; B[2,2*k+1]=b[k]
    return B, A

nx, ny, L, H, nu, P = 20, 10, 1.0, 0.2, 0.3, -1.0e3
E = torch.tensor(2.1e11, requires_grad=True)
coords, elems = build_mesh(nx, ny, L, H)
D = D_plane_strain(E, nu)
rows, cols, vals = [], [], []
for tri in elems:
    idx = tri.tolist(); ce = coords[idx]
    B, A = cst_BA(ce); Ke = (B.T @ D @ B) * A
    dof = [2*idx[0],2*idx[0]+1,2*idx[1],2*idx[1]+1,2*idx[2],2*idx[2]+1]
    for a in range(6):
        for b_ in range(6):
            rows.append(dof[a]); cols.append(dof[b_]); vals.append(Ke[a,b_])
indices = torch.tensor([rows, cols], dtype=torch.long)
values = torch.stack(vals)
n_dof = coords.shape[0]*2

left = [i for i in range(coords.shape[0]) if coords[i,0] == 0.0]
fixed = [d for n_ in left for d in (2*n_, 2*n_+1)]
penalty = values.abs().max().item() * 1e8
extra_idx = torch.tensor([fixed, fixed], dtype=torch.long)
extra_val = torch.full((len(fixed),), penalty, dtype=values.dtype)
indices_p = torch.cat([indices, extra_idx], dim=1)
values_p = torch.cat([values, extra_val])

tip = nx*(ny+1) + ny//2
f = torch.zeros(n_dof); f[2*tip+1] = P
u = SparseSolveAutograd.apply(indices_p, values_p, f, n_dof)
u_tip = u[2*tip+1]
I = H**3/12.0; delta_eb = P*L**3/(3*E.detach().item()*I)
print(f"FE tip displacement     : {u_tip.detach().item():.6e} m")
print(f"Euler-Bernoulli analytic: {delta_eb:.6e} m  (err {(u_tip.detach().item()-delta_eb)/delta_eb*100:+.1f}%)")
u_tip.abs().backward()
print(f"d|u_tip|/dE             : {E.grad.item():.6e}")

## 4. Generalized-α dynamics

Hulbert–Chung gen-α with `rho_inf=0.5` numerically dissipates a high-frequency mode while preserving the low-frequency motion; `rho_inf=1.0` (Newmark-β) conserves both. We test on a 2-DOF oscillator (low ω = 2π, high ω = 2π×10³).

In [ ]:
import math, torch
import matplotlib.pyplot as plt
from time_integrators import gen_alpha_params, gen_alpha_step

def energy(u, v, K, M):
    return 0.5*M*v*v + 0.5*K*u*u

def run(rho_inf, n_steps=100, dt=1e-3):
    M = torch.tensor([1., 1.], dtype=torch.float64)
    K = torch.tensor([(2*math.pi)**2, (2*math.pi*1e3)**2], dtype=torch.float64)
    u = torch.tensor([1., 1.], dtype=torch.float64); v = torch.zeros(2, dtype=torch.float64)
    a = -K*u/M; f = torch.zeros(2, dtype=torch.float64)
    am, af, beta, gamma = gen_alpha_params(rho_inf)
    Es = torch.empty(n_steps+1, 2, dtype=torch.float64); Es[0] = energy(u,v,K,M)
    for n in range(n_steps):
        u, v, a = gen_alpha_step(u, v, a, K, M, f, dt, am, af, beta, gamma)
        Es[n+1] = energy(u,v,K,M)
    return Es

n_steps, dt = 100, 1e-3
t = torch.arange(n_steps+1) * dt
Es10 = run(1.0, n_steps, dt); Es05 = run(0.5, n_steps, dt)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(t.numpy(), (Es10[:,1]/Es10[0,1]).numpy(), '-', label=r'$\rho_\infty=1.0$ (conserving)')
ax.plot(t.numpy(), (Es05[:,1]/Es05[0,1]).numpy(), '--', label=r'$\rho_\infty=0.5$ (dissipative)')
ax.set_xlabel('time'); ax.set_ylabel('high-freq mode energy / initial')
ax.set_ylim(-0.05, 1.15); ax.legend(frameon=False); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"final HF energy ratio  rho=1.0: {float(Es10[-1,1]/Es10[0,1]):.4f}")
print(f"final HF energy ratio  rho=0.5: {float(Es05[-1,1]/Es05[0,1]):.4e}")

## 5. Mixed-precision CG

`cg_mixed_precision` runs the inner Krylov iterations in float32 and refines the solution back to float64 with one or more correction sweeps. On ill-conditioned 1D Laplacians this delivers float64 accuracy at near-float32 speed.

In [ ]:
import time, torch
from mixed_precision_cg import cg_mixed_precision

torch.manual_seed(0)
n = 5000
idx_main, idx_off = torch.arange(n), torch.arange(n-1)
rows = torch.cat([idx_main, idx_off, idx_off+1])
cols = torch.cat([idx_main, idx_off+1, idx_off])
vals = torch.cat([2.*torch.ones(n), -torch.ones(n-1), -torch.ones(n-1)])
indices = torch.stack([rows, cols])
K64 = torch.sparse_coo_tensor(indices, vals.double(), (n, n)).coalesce()
K32 = torch.sparse_coo_tensor(indices, vals.float(), (n, n)).coalesce()

def matvec(v):
    K = K32 if v.dtype == torch.float32 else K64
    return torch.sparse.mm(K, v.unsqueeze(1)).squeeze(1)

b = torch.randn(n, dtype=torch.float64)
for prec, tol in [('float64', 1e-10), ('float32', 1e-6), ('mixed', 1e-10)]:
    rhs = b.float() if prec == 'float32' else b
    t0 = time.perf_counter()
    x, it, conv = cg_mixed_precision(matvec, rhs, tol=tol, max_iter=20000, precision=prec)
    dt = time.perf_counter() - t0
    res = float(torch.linalg.norm(torch.sparse.mm(K64, x.double().unsqueeze(1)).squeeze(1) - b))
    print(f"{prec:<8}  time={dt:7.3f}s  iters={it:5d}  ||Kx-b||={res:.3e}  converged={conv}")

## 6. Speedup curve

Direct sparse solve (SciPy SuperLU) vs CG on the quasi-static Miehe tension benchmark, taken from `examples/quasistatic/miehe_tension/timing_speedup.csv`.

In [ ]:
import matplotlib.pyplot as plt

# Verified numbers (commit on main, CPU)
dofs    = [1220, 4414, 8954, 15846, 31420]
speedup = [2.00, 8.75, 8.36, 10.37, 9.51]

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(dofs, speedup, 'o-', color='#2c7fb8', lw=1.6, ms=6)
ax.set_xscale('log'); ax.set_xlabel('DOFs'); ax.set_ylabel('speedup (CG / SciPy direct)')
ax.set_title('Quasi-static Miehe tension: direct vs CG')
ax.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()

## 7. Where next

- Documentation: <https://cems-lab.github.io/PhAST/>
- Issue tracker: <https://github.com/CEMS-Lab/PhAST/issues>
- More demos to explore in the cloned repo:
  - `examples/solid_mechanics/neohookean_plate/run.py` — nonlinear Newton with autograd-through-solve
  - `examples/quasistatic/miehe_tension/` — full Miehe tension benchmark
  - `examples/dynamic/kalthoff/` — dynamic Kalthoff–Winkler impact
  - `paper/` — CMAME paper sources (build with `latexmk -pdf main`)